In [2]:
import csv
import re
import polars as pl
from rapidfuzz import process, fuzz
from collections import defaultdict
import random
import re
import lzma
import numpy as np
from scipy.sparse import csr_matrix

In [3]:
ratings = (
    pl.read_csv("../datasets/movieLense-100k/ratings.csv")
    .select(["userId", "movieId", "rating"])
    .with_columns(
        pl.col("userId").cast(pl.Int32),
        pl.col("movieId").cast(pl.Int32),
        pl.col("rating").cast(pl.Float32),
    )
)

users = ratings["userId"].to_numpy()
movies = ratings["movieId"].to_numpy()
stars = ratings["rating"].to_numpy()

In [4]:
print(movies)

[     1      3      6 ... 168250 168252 170875]


In [12]:
#movieId,title,genres

class movie(object):
    def __init__(self,movie_id,title,generes):
        self.movie_id = movie_id
        self.title = title
        self.generes = generes
        self.plot_summary = None
        # can extend this into the tags as well


In [21]:
# enriched MovieLens-32M
df = pl.read_parquet("hf://datasets/krishnakamath/movielens-32m-movies-enriched/data/train-00000-of-00001.parquet")


In [22]:
def get_year(title):
    match = re.search(r"\((\d{4})\)\s*$", title)
    return match.group(1) if match else None


def clean_title(title):
    title = re.sub(r"\(\d{4}\)\s*$", "", title)
    title = re.sub(r"\(a\.k\.a\..*?\)", "", title, flags=re.I)
    title = re.sub(r"[^a-z0-9]+", " ", title.lower())
    return " ".join(title.split())

In [23]:
exact_lookup = dict(zip(df["title"].to_list(), df["plot_summary"].to_list()))

year_lookup = {}

for title, summary in zip(df["title"].to_list(), df["plot_summary"].to_list()):
    if summary is None:
        continue

    year = get_year(title)

    if year is not None:
        year_lookup.setdefault(year, {})
        year_lookup[year][clean_title(title)] = summary

In [24]:
movies_information = []

with open("../week1/movieLense-100k/movies.csv", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        new_movie = movie(row["movieId"], row["title"], row["genres"])
        new_movie.plot_summary = exact_lookup.get(new_movie.title)
        movies_information.append(new_movie)

In [29]:
exact_lookup = dict(zip(df["title"].to_list(), df["plot_summary"].to_list()))

for m in movies_information:
    m.plot_summary = exact_lookup.get(m.title)

In [30]:
matched = sum(m.plot_summary is not None for m in movies_information)

print(f"Matched: {matched}/{len(movies_information)}")
print(f"Coverage: {matched / len(movies_information):.2%}")

Matched: 9464/9742
Coverage: 97.15%


In [39]:
users = ratings["userId"].to_numpy()
movies = ratings["movieId"].to_numpy()
stars = ratings["rating"].to_numpy()

R = csr_matrix(
    (stars, (users, movies)),
    shape=(users.max() + 1, movies.max() + 1),
    dtype=np.float32,
)

R_user = R
R_movie = R.tocsc()

In [ ]:
def enclosing_subgraph(u, v, h, R_user, R_movie):
    U = {u}
    V = {v}

    U_fringe = {u}
    V_fringe = {v}

    for i in range(h):

        U_new = set()

        for movie in V_fringe:
            start = R_movie.indptr[movie]
            end = R_movie.indptr[movie + 1]

            U_new.update(R_movie.indices[start:end])

        U_new -= U

        V_new = set()

        for user in U_fringe:
            start = R_user.indptr[user]
            end = R_user.indptr[user + 1]

            V_new.update(R_user.indices[start:end])

        V_new -= V

        U_fringe = U_new
        V_fringe = V_new

        U |= U_fringe
        V |= V_fringe


    edges = []

    for user in U:
        start = R_user.indptr[user]
        end = R_user.indptr[user + 1]

        user_movies = R_user.indices[start:end]
        user_ratings = R_user.data[start:end]

        for movie, rating in zip(user_movies, user_ratings):

            movie = int(movie)

            if movie not in V:
                continue

            if user == u and movie == v:
                continue

            edges.append((user, movie, float(rating)))


    return {
        "users": U,
        "movies": V,
        "edges": edges,
    }

In [46]:
G = enclosing_subgraph(
    u=1,
    v=296,
    h=1,
    R_user=R_user,
    R_movie=R_movie,
)

In [48]:
print("Users:", G["users"])
print("Movies:", G["movies"])
print("Edges:", G["edges"][:20])

Users: {1, np.int32(4), np.int32(5), np.int32(6), np.int32(8), np.int32(10), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(21), np.int32(23), np.int32(24), np.int32(26), np.int32(28), np.int32(29), np.int32(32), np.int32(33), np.int32(37), np.int32(38), np.int32(39), np.int32(40), np.int32(41), np.int32(42), np.int32(43), np.int32(45), np.int32(50), np.int32(54), np.int32(56), np.int32(57), np.int32(58), np.int32(62), np.int32(63), np.int32(64), np.int32(66), np.int32(67), np.int32(68), np.int32(69), np.int32(72), np.int32(74), np.int32(76), np.int32(78), np.int32(81), np.int32(84), np.int32(91), np.int32(94), np.int32(96), np.int32(99), np.int32(100), np.int32(102), np.int32(103), np.int32(105), np.int32(107), np.int32(109), np.int32(110), np.int32(115), np.int32(117), np.int32(118), np.int32(119), np.int32(121), np.int32(122), np.int32(123), np.int32(124), np.int32(125), np.int32(126), np.int32(130), np.int32(131), np.int32(132), np.int32(133), np.int

In [49]:
print(R_user)

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 100836 stored elements and shape (611, 193610)>
  Coords	Values
  (1, 1)	4.0
  (1, 3)	4.0
  (1, 6)	4.0
  (1, 47)	5.0
  (1, 50)	5.0
  (1, 70)	3.0
  (1, 101)	5.0
  (1, 110)	4.0
  (1, 151)	5.0
  (1, 157)	5.0
  (1, 163)	5.0
  (1, 216)	5.0
  (1, 223)	3.0
  (1, 231)	5.0
  (1, 235)	4.0
  (1, 260)	5.0
  (1, 296)	3.0
  (1, 316)	3.0
  (1, 333)	5.0
  (1, 349)	4.0
  (1, 356)	4.0
  (1, 362)	5.0
  (1, 367)	4.0
  (1, 423)	3.0
  (1, 441)	4.0
  :	:
  (610, 156371)	5.0
  (610, 156726)	4.5
  (610, 157296)	4.0
  (610, 158238)	5.0
  (610, 158721)	3.5
  (610, 158872)	3.5
  (610, 158956)	3.0
  (610, 159093)	3.0
  (610, 160080)	3.0
  (610, 160341)	2.5
  (610, 160527)	4.5
  (610, 160571)	3.0
  (610, 160836)	3.0
  (610, 161582)	4.0
  (610, 161634)	4.0
  (610, 162350)	3.5
  (610, 163937)	3.5
  (610, 163981)	3.5
  (610, 164179)	5.0
  (610, 166528)	4.0
  (610, 166534)	4.0
  (610, 168248)	5.0
  (610, 168250)	5.0
  (610, 168252)	5.0
  (610, 170875)	3.0


In [51]:
# user's movies:
def users_movies(user):
    # user here being a number
    return list(zip(R_user.indices[R_user.indptr[user]:R_user.indptr[user+1]], R_user.data[R_user.indptr[user]:R_user.indptr[user+1]]))

In [ ]:
user_1_movies = users_movies(1)

[(np.int32(1), np.float32(4.0)),
 (np.int32(3), np.float32(4.0)),
 (np.int32(6), np.float32(4.0)),
 (np.int32(47), np.float32(5.0)),
 (np.int32(50), np.float32(5.0)),
 (np.int32(70), np.float32(3.0)),
 (np.int32(101), np.float32(5.0)),
 (np.int32(110), np.float32(4.0)),
 (np.int32(151), np.float32(5.0)),
 (np.int32(157), np.float32(5.0)),
 (np.int32(163), np.float32(5.0)),
 (np.int32(216), np.float32(5.0)),
 (np.int32(223), np.float32(3.0)),
 (np.int32(231), np.float32(5.0)),
 (np.int32(235), np.float32(4.0)),
 (np.int32(260), np.float32(5.0)),
 (np.int32(296), np.float32(3.0)),
 (np.int32(316), np.float32(3.0)),
 (np.int32(333), np.float32(5.0)),
 (np.int32(349), np.float32(4.0)),
 (np.int32(356), np.float32(4.0)),
 (np.int32(362), np.float32(5.0)),
 (np.int32(367), np.float32(4.0)),
 (np.int32(423), np.float32(3.0)),
 (np.int32(441), np.float32(4.0)),
 (np.int32(457), np.float32(5.0)),
 (np.int32(480), np.float32(4.0)),
 (np.int32(500), np.float32(3.0)),
 (np.int32(527), np.float32(5

In [62]:
def get_user_movie_bins(u, R_user, movie_df):
    start, end = R_user.indptr[u], R_user.indptr[u + 1]

    user_ratings = pl.DataFrame({
        "movie_id": R_user.indices[start:end],
        "rating": R_user.data[start:end],
    }).with_columns(
        pl.when(pl.col("rating") >= 4).then(pl.lit("5-4"))
        .when(pl.col("rating") >= 3).then(pl.lit("4-3"))
        .when(pl.col("rating") >= 2).then(pl.lit("3-2"))
        .when(pl.col("rating") >= 1).then(pl.lit("2-1"))
        .otherwise(pl.lit("below-1"))
        .alias("rating_bin")
    )

    return user_ratings.join(movie_df, on="movie_id", how="left").sort("rating", descending=True)

In [69]:
new_rating = get_user_movie_bins(1, R_user, df)

In [91]:
new_rating = new_rating.filter(pl.col("movie_id") != 70)

In [92]:
def concat_movie_text(user_movies):
    text_cols = [c for c, dtype in zip(user_movies.columns, user_movies.dtypes) if dtype == pl.String]
    return "\n".join(" ".join(str(v) for v in row if v is not None) for row in user_movies.select(text_cols).iter_rows())

In [93]:
texts = {
    bin_name: concat_movie_text(new_rating.filter(pl.col("rating_bin") == bin_name))
    for bin_name in ["5-4", "4-3", "3-2", "2-1"]
}

In [94]:
print(texts)

{'5-4': "5-4 Seven (a.k.a. Se7en) (1995) Seven, also known as Se7en, is a psychological thriller directed by David Fincher. The film follows two detectives, a rookie and a veteran, as they hunt down a serial killer who uses the seven deadly sins as his modus operandi. As the detectives get closer to the killer, they are drawn into a dark and twisted game that challenges their beliefs and morals. David Fincher\n5-4 Usual Suspects, The (1995) The Usual Suspects is a neo-noir mystery film directed by Bryan Singer. The movie follows the interrogation of a conman named Verbal Kint, who recounts the events leading up to a deadly heist orchestrated by the mysterious criminal mastermind Keyser Söze. As the pieces of the puzzle come together, the truth becomes increasingly elusive. Bryan Singer\n5-4 Bottle Rocket (1996) 'Bottle Rocket' is a quirky comedy from 1996 that follows a group of eccentric friends who embark on a series of heists in an attempt to break free from their mundane lives. As 

In [98]:
encoded_data_5_4 = texts["5-4"].encode("utf-8")
raw_compress_5_4 = lzma.compress(encoded_data_5_4)
print(raw_compress_5_4.__sizeof__())

encoded_data_4_3 = texts["4-3"].encode("utf-8")
raw_compress_4_3 = lzma.compress(encoded_data_4_3)
print(raw_compress_4_3.__sizeof__())

encoded_data_3_2 = texts["3-2"].encode("utf-8")
raw_compress_3_2 = lzma.compress(encoded_data_3_2)
print(raw_compress_3_2.__sizeof__())

encoded_data_2_1 = texts["2-1"].encode("utf-8")
raw_compress_2_1 = lzma.compress(encoded_data_2_1)
print(raw_compress_2_1.__sizeof__())

26233
4517
1301
385


In [99]:
to_test = movies_information[69]
new_info = f"{to_test.genres} + {to_test.plot_summary} + {to_test.title}"
print(new_info)

Documentary + A documentary exploring the life and career of Nico, a singer, model, and actress known for her work with the Velvet Underground and her solo music career. The film delves into Nico's complex personality, her struggles with addiction, and her impact on the music industry. + Nico Icon (1995)


In [102]:
# so if 3B1B is right user 1, to movie 70 should be the most correct to be 3 stars.
new_encoded_data_5_4 = f"{texts['5-4']} {new_info}".encode("utf-8")
new_encoded_data_5_4 = lzma.compress(new_encoded_data_5_4)
print(new_encoded_data_5_4.__sizeof__()/raw_compress_5_4.__sizeof__())

new_encoded_data_4_3 = f"{texts['4-3']} {new_info}".encode("utf-8")
new_compressed_data_4_3 = lzma.compress(new_encoded_data_4_3)
print(new_compressed_data_4_3.__sizeof__()/raw_compress_4_3.__sizeof__())

new_encoded_data_3_2 = f"{texts['3-2']} {new_info}".encode("utf-8")
new_compressed_data_3_2 = lzma.compress(new_encoded_data_3_2)
print(new_compressed_data_3_2.__sizeof__()/raw_compress_3_2.__sizeof__())

new_encoded_data_2_1 = f"{texts['2-1']} {new_info}".encode("utf-8")
new_compressed_data_2_1 = lzma.compress(new_encoded_data_2_1)
print(new_compressed_data_2_1.__sizeof__()/raw_compress_2_1.__sizeof__())

1.0039644722296344
1.0274518485720612
1.1168332052267487
1.4467532467532467
